# 02 Pipeline

A compact, cloneable pipeline: Read prep resolves source identity and Lineage, physical IO executes, and checks plus metadata registration stay visible.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Load the shared Fabric configuration, public APIs, and only the two registries needed across Read blocks.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_source_stability,
    check_schema,
    observe_table,
    profile_and_register_table,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    write_lakehouse_table,
    write_pipeline_prep,
    widget_select_data_contract,
    widget_view_catalogue,
)

READ_PREPS = {}
READ_DFS = {}

## Notebook controls

Use one optional Data Contract selector and one Catalogue view. FabricOps uses the notebook name as the stable logical Lineage identity across environments; runtime notebook, workspace, and environment IDs remain audit context.

In [ ]:
# Step 2: leave False for the first run, before Lineage and frozen contracts exist.
# Step 4: set True to select one frozen contract for every lineage-linked table_id.
VALIDATE_DATA_CONTRACTS = False
CONTRACT_SELECTION = (
    widget_select_data_contract(spark_session=spark)
    if VALIDATE_DATA_CONTRACTS
    else None
)

catalogue_widget = widget_view_catalogue(mode="explore", spark_session=spark)

## How to read the blocks

Each Read block defines physical source identity. Each Write block identifies one governed target `table_id`; FabricOps resolves its frozen load strategy and validates that this notebook owns the target.

In [ ]:
# One governed target table_id has one owning pipeline/notebook writer.
WRITE_TABLE_ID = "replace-with-governed-target-table-id"


# R. Read

Each Read follows **define → prep → physical read → data checks → profile/register → store → view**.

## READ 1 — Orders

A complete Lakehouse source read with source identity and Lineage resolved by prep.

In [ ]:
READ = 1
READ_NAME = "Orders"
READ_TARGET = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"

In [ ]:
read_prep = read_pipeline_prep(
    source_target=READ_TARGET,
    source_schema=READ_SCHEMA,
    source_table=READ_TABLE,
)
READ_TABLE_ID = read_prep["table_id"]

read_df = read_lakehouse_table(
    table_id=READ_TABLE_ID,
    spark_session=spark,
)
check_schema(
    READ_TABLE_ID,
    dataframe=read_df,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)

if VALIDATE_DATA_CONTRACTS:
    observation = observe_table(
        READ_TABLE,
        target=READ_TARGET,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(
        observation,
        table_id=READ_TABLE_ID,
        raise_on_failure=True,
    )
    check_source_stability(
        observation,
        target_table_id=WRITE_TABLE_ID,
    )
check_dq(
    read_df,
    table_id=READ_TABLE_ID,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)
read_profile = profile_and_register_table(
    read_df,
    profile_role="source",
    table=read_prep["source"],
)
display(read_profile)

READ_PREPS[READ] = read_prep
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

## READ 2 — Products

A complete Lakehouse reference read with the same visible stages.

In [ ]:
READ = 2
READ_NAME = "Products"
READ_TARGET = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"

In [ ]:
read_prep = read_pipeline_prep(
    source_target=READ_TARGET,
    source_schema=READ_SCHEMA,
    source_table=READ_TABLE,
)
READ_TABLE_ID = read_prep["table_id"]

read_df = read_lakehouse_table(
    table_id=READ_TABLE_ID,
    spark_session=spark,
)
check_schema(
    READ_TABLE_ID,
    dataframe=read_df,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)

if VALIDATE_DATA_CONTRACTS:
    observation = observe_table(
        READ_TABLE,
        target=READ_TARGET,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(
        observation,
        table_id=READ_TABLE_ID,
        raise_on_failure=True,
    )
    check_source_stability(
        observation,
        target_table_id=WRITE_TABLE_ID,
    )
check_dq(
    read_df,
    table_id=READ_TABLE_ID,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)
read_profile = profile_and_register_table(
    read_df,
    profile_role="source",
    table=read_prep["source"],
)
display(read_profile)

READ_PREPS[READ] = read_prep
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

## READ 3 — Order History

The SQL stays visible and executes in the Warehouse; its aggregate receives a diagnostic profile.

In [ ]:
READ = 3
READ_NAME = "Order History"
READ_TARGET = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"

READ_QUERY = """
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM demo.order_history
GROUP BY customer_id
"""

In [ ]:
read_prep = read_pipeline_prep(
    source_target=READ_TARGET,
    source_schema=READ_SCHEMA,
    source_table=READ_TABLE,
)
READ_TABLE_ID = read_prep["table_id"]

read_df = read_warehouse_query(
    READ_QUERY,
    target=read_prep["source"]["target"],
    spark_session=spark,
)
check_schema(
    READ_TABLE_ID,
    dataframe=read_df,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)

if VALIDATE_DATA_CONTRACTS:
    observation = observe_table(
        READ_TABLE,
        target=READ_TARGET,
        schema=READ_SCHEMA,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(
        observation,
        table_id=READ_TABLE_ID,
        raise_on_failure=True,
    )
    check_source_stability(
        observation,
        target_table_id=WRITE_TABLE_ID,
    )
check_dq(
    read_df,
    table_id=READ_TABLE_ID,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)
read_profile = profile_and_register_table(
    read_df,
    profile_role="source",
    table=read_prep["source"],
    complete_table=False,
)
display(read_profile)

READ_PREPS[READ] = read_prep
READ_DFS[READ] = read_df
catalogue_widget["show"](table_id=READ_TABLE_ID)

### Profile semantics

Complete physical reads update canonical `METADATA_DATA_CATALOGUE`, `METADATA_DATA_PROFILED`, and eligible `METADATA_DATA_PROFILED_FREQUENCY` snapshots. Incremental subsets and custom query results return diagnostic profiles without replacing canonical full-table metadata. Source and target lineage participation is completed during write preparation.

# T. Transform

**Business transformation is project-owned PySpark.** FabricOps does not hide these joins or calculations.

In [ ]:
orders_df = READ_DFS[1].alias("orders")
products_df = READ_DFS[2].alias("products")
history_df = READ_DFS[3].alias("history")

transformed_df = (
    orders_df
    .join(products_df, on="product_id", how="left")
    .join(history_df, on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)
display(transformed_df)

# W. Write

## WRITE 1 — Curated Orders

The Write follows **identify governed target → resolve frozen load strategy → checks → physical write → published read → profile/register → view**.

In [ ]:
WRITE = 1
WRITE_NAME = "Curated Orders"
write_df = transformed_df

In [ ]:
write_prep = write_pipeline_prep(
    write_df,
    target_table_id=WRITE_TABLE_ID,
    source_preps=[
        READ_PREPS[1],
        READ_PREPS[2],
        READ_PREPS[3],
    ],
)
prepared_df = write_prep["df"].persist()

In [ ]:
check_schema(
    WRITE_TABLE_ID,
    dataframe=write_df,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)
check_dq(
    write_df,
    table_id=WRITE_TABLE_ID,
    enabled=VALIDATE_DATA_CONTRACTS,
    raise_on_failure=True,
)

### Physical write

`repartition_by=4` uses distributed Spark execution; it does not create Python threads or independent writers.

In [ ]:
write_lakehouse_table(
    prepared_df,
    write_prep["target"]["table_name"],
    target=write_prep["target"]["target"],
    schema=write_prep["target"]["schema"],
    mode=write_prep["mode"],
    options=write_prep["options"],
    load_strategy=write_prep["load_strategy"],
    load_strategy_parameters=write_prep["load_strategy_parameters"],
    processing_scope=write_prep["scope"],
    success_context=(write_prep["success_context"] if VALIDATE_DATA_CONTRACTS else None),
    repartition_by=4,
)
prepared_df.unpersist()

### Published state and stored metadata

Read the published table back before recording its canonical full-table profile.

In [ ]:
published_df = read_lakehouse_table(
    table_id=WRITE_TABLE_ID,
    spark_session=spark,
)
write_profile = profile_and_register_table(
    published_df,
    profile_role="target",
    table=write_prep["target"],
    load_strategy=write_prep["load_strategy"],
    load_strategy_parameters=write_prep["load_strategy_parameters"],
)
display(write_profile)
catalogue_widget["show"](table_id=WRITE_TABLE_ID)